In [1]:
import psycopg2
import numpy as np
import psycopg2.extras as extras
import pandas as pd
import pickle
from preprocessor import preprocessor
from config import pathconfig
from tqdm import tqdm
from multiprocessing import Pool

In [2]:
def load_pkl(pkl_file):
    with open(pkl_file, 'rb') as inp:
        print(f"loading pickle file: {pkl_file}")
        pickled_obj = pickle.load(inp)
    return pickled_obj

In [3]:
if pathconfig.PRE_PKL_PATH.is_file():
    print('Loading PKL file...')
    pre = load_pkl(pathconfig.PRE_PKL_PATH)
    print('Successfully Loaded Pickle file')
else:
    print("NO PICKLE FILE FOUND SAD :(")

Loading PKL file...
loading pickle file: D:\recommendation_news_api\repository\data-processing\processed_data\train_data\pre.pkl
Successfully Loaded Pickle file


In [4]:
pre.article_df

,user,url,additional_images,content,date,head_image,heading,label,source,author,...,all_heading_content,keywords,Dominant_Topic,final_rating,user_index,url_index,label_index,article_id,user_keywords,keywords_words
0,db783e901db42f2290736d314d08bdea,https://www.ratopati.com/story/94744,['https://ratopati.prixa.net/media/albums/666_...,['नेपाल राष्ट्र बैंकका अनुसार आज (साउन ७) एक अ...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/Exchan...,यस्तो छ आजको विनिमयदरः अमेरिकी डलरदेखि कतारी र...,अर्थ,ratopati,रातोपाटी संवाददाता,...,यस्तो विनिमयदरः अमेरि कतारी रियालसम्म नेपाल रा...,"[13073, 27456, 12617, 6083, 31465, 19698, 1293...",1,2.723818,158790,31534,2,ekuIAXIip3isHjFZAdmOs2rgb4KgToUZSIvURG0_AoA,"[पैसा, रुपैयाँ, बिक्रीदर, खरिददर, कतारी, विनिम...",पैसा रुपैयाँ बिक्रीदर खरिददर कतारी विनिमयदरः अ...
1,83ba6e048bfb2245baeff4a8c40a7372,https://www.ratopati.com/story/94749,[],['बलिउड नायिका सुष्मिता सेनको एउटा फोटो इन्टरन...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/sushmi...,"सुस्मिता सेनलाई चुम्बन गरिरहेको भेटिए प्रेमी, ...",मनोरञ्जन,ratopati,रातोपाटी संवाददाता,...,"सुस्मिता चुम्बन गरिरह भेटिए प्रेमी, फोटो चर्चि...","[5968, 30411, 14608, 14843, 9248, 1574, 8248, ...",3,1.520255,95385,31538,20,yJC-tHpfRpKEH3En-YScXK0m8eEWOqouPijF8BonHKI,"[सुष्मिता, रोहमन, छुट्टी, तस्बिर, चुम्बन, प्रे...","सुष्मिता रोहमन छुट्टी तस्बिर चुम्बन प्रेमी, मु..."
2,a704f987e0bc79f6bc185228f1cdfe8a,https://www.ratopati.com/story/94766,[],['काठमाडौं– प्रतिवन्धित नेपाल कम्युनिस्ट पार्ट...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/biplab...,"सरकार वार्ताको पक्षमा देखिएन, देशलाई युद्धमा ध...",मुख्य समाचार,ratopati,रातोपाटी संवाददाता,...,"सरकार वार्ता पक्ष देखिएन, युद्ध धकेल्न खोज्योः...","[27982, 34743, 30577, 20802, 28810, 27104, 253...",2,1.909831,120855,31547,21,PkP3AJocUXbZCOQBCOkAACEZ6TwuXPxSMHJxzZQjN5U,"[पक्ष, विवाद, पार्टी, अनुसार, संयुक्त, बैठक, आ...","विप्लव वार्ता युद्ध जबरजस्त धकेल्न शान्ति, सरक..."
3,a7c27e8ad54f246d33d92c85921c21eb,https://www.ratopati.com/story/94769,[],['काठमाडौं– छुट्टछुट्टै दुर्घटनामा परेर सर्लाह...,2076-04-07 00:00:00+05:45,https://ratopati.prixa.net/media/albums/death_...,सर्लाहीमा छुट्टाछुट्टै दुर्घटनामा दुई जनाको मृ...,प्रदेश मुख्य समाचार,ratopati,रातोपाटी संवाददाता,...,सर्लाही छुट्टाछुट्टै दुर्घटना मृत्यु काठमाडौं–...,"[11916, 31053, 32043, 21744, 8895, 162, 10331,...",5,1.836916,121370,31549,13,2gafeJYipiE31yBVKVOTjGN2bCdzFUPHZ8ymiWoiIvk,"[लागि, पार्टी, तयार, सरकार, प्रधानमन्त्री, दुर...",मृत्यु दुर्घटना घाइते उपचार बालक मोटरसाइकल लाल...
4,1c58fbb11cd5d6aa48c87b9d5404fddb,https://www.ratopati.com/story/94721,[],['बुटवल-अनैतिक सम्बन्ध राखेको आरोपमा नवलपरासीक...,2076-04-06 00:00:00+05:45,https://ratopati.prixa.net/media/albums/arrest...,होटलमा अनैतिक सम्बन्ध राख्ने १० जना पक्राउ,समाचार,ratopati,रातोपाटी संवाददाता,...,होटल अनैतिक सम्बन्ध राख् पक्राउ बुटवल-अनैतिक स...,"[36721, 14183, 36254, 15011, 18182, 5073, 6950...",0,1.391775,20538,31529,31,UhK8tdrfsbFHVGSSnfApmu37DePbCDPedGr2whVu2AI,"[होटेल, होटल, अनैतिक, पक्राउ, नवलपरासी, सञ्चाल...",होटेल होटल अनैतिक पक्राउ नवलपरासी सञ्चालक प्रह...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38049,5cb16210f5bc5f18bb852ab846f3b5a2,https://www.thahakhabar.com/news/127679,[],['\xa0नेकपा एमालेका अध्यक्ष एवं प्रधानमन्त्री ...,2078-02-24 00:00:00+05:45,https://thahacdn.prixacdn.net/media/albums/Kha...,खनाल-नेपाल समूहका नेता सरकारमा सहभागी हुँदै !,राजनीति,thahakhabar,थाहा संवाददाता,...,खनाल-नेपाल समूह नेता सरकार सहभागी नेकपा एमाल अ...,"[13635, 1725, 19919, 25420, 8942, 5742, 31687,...",11,2.876623,67476,35675,22,1GFRtO4g4f79IRJjGdkvTisLdsMCtpQsVWBFDaW0Vec,"[खनाल–नेपाल, समूह, मन्त्रालय, मन्त्रिमण्डल, प्...",खनाल–नेपाल समूह मन्त्रालय मन्त्रिमण्डल प्रधानम...
38050,93c95988ec06dc1569a02ccb931c43a4,https://www.thahakhabar.com/news/127379,[],['\xa0नारायणहिटी\xa0राजदरबार हत्याकाण्ड छानबिन...,2078-02-20 00:00:00+05:45,https://thahacdn.prixacdn.net/media/albums/lin...,बाबुरामप्रति लिङ्देनको कटाक्ष

In [4]:
pre.article_df.keywords_words

0        पैसा रुपैयाँ बिक्रीदर खरिददर कतारी विनिमयदरः अ...
1        सुष्मिता रोहमन छुट्टी तस्बिर चुम्बन प्रेमी, मु...
2        विप्लव वार्ता युद्ध जबरजस्त धकेल्न शान्ति, सरक...
3        मृत्यु दुर्घटना घाइते उपचार बालक मोटरसाइकल लाल...
4        होटेल होटल अनैतिक पक्राउ नवलपरासी सञ्चालक प्रह...
                               ...                        
38049    खनाल–नेपाल समूह मन्त्रालय मन्त्रिमण्डल प्रधानम...
38050    ज्ञानेन्द्र भ्रम राजा अन्याय भट्टराई कटाक्ष तप...
38051    कांग्रेस माओवा मन्त्रालय रामसरोज ओमप्रकाश वीरे...
38052    हजार करोड रसिया, विश्व कोरोना ब्राजिल, इटाली, ...
38053    सुदूरपश्चिम भट्ट लेखराज प्रदेशसभा एमाल पक्ष प्...
Name: keywords_words, Length: 38054, dtype: object

In [7]:
pre.article_df.article_id.iloc[0]

'ekuIAXIip3isHjFZAdmOs2rgb4KgToUZSIvURG0_AoA'

In [6]:
pre.article_df.keywords_words.iloc[0]

'पैसा रुपैयाँ बिक्रीदर खरिददर कतारी विनिमयदरः अमेरि रियालसम्म यस्तै दिराम युएई पाउण्ड रियाल युरो यसैगरी यस्तो राष्ट्र बैंक अनुसार नेपाल'

पैसा
रुपैयाँ
बिक्रीदर
खरिददर
कतारी
विनिमयदरः
अमेरि
रियालसम्म
यस्तै
दिराम
युएई
पाउण्ड
रियाल
युरो
यसैगरी
यस्तो
राष्ट्र
बैंक
अनुसार
नेपाल


In [4]:
pre.article_df.keywords_words.values

array(['पैसा रुपैयाँ बिक्रीदर खरिददर कतारी विनिमयदरः अमेरि रियालसम्म यस्तै दिराम युएई पाउण्ड रियाल युरो यसैगरी यस्तो राष्ट्र बैंक अनुसार नेपाल',
       'सुष्मिता रोहमन छुट्टी तस्बिर चुम्बन प्रेमी, मुस्कुराइरह फोटो सुस्मिता भेटिए सेल्फी गाला आफ्ना होला, नायि मिडियामार्फत् इन्टरनेट मनाइरह उत्सव एउटा',
       'विप्लव वार्ता युद्ध जबरजस्त धकेल्न शान्ति, सरकार पक्ष कार्यकर्ता समृद्धि लगाएर देखिएन, कुप्रचार झूठो स्पष्टै झूटा नरहेर प्रतिवन्धित बढावा सकिन्छ’,',
       ...,
       'कांग्रेस माओवा मन्त्रालय रामसरोज ओमप्रकाश वीरेन्द्र मन्त्री सहभागी सिंह सरकार यादव प्रदेश तर्फबाट बन्दैछन्। सिँचाई मन्त्रालय, केन्द्र गरेकाे फुटाएर लागिसक',
       'हजार करोड रसिया, विश्व कोरोना ब्राजिल, इटाली, बेलायत, छन्। मुलुक कोरोनाबाट संक्रमित पेरु, थिए। जहाँ हजार, अछुतो टर्की, उच्च हालसम्म',
       'सुदूरपश्चिम भट्ट लेखराज प्रदेशसभा एमाल पक्ष प्रदेश कमिटी रावल माओवा समूह प्रभाव असफल फिर्ता संसदीय मतदान मन्त्री बोहरा, स्पष्टीकरण पटक–पटक'],
      dtype=object)

In [5]:
allkeywords = " ". join(pre.article_df.keywords_words.values)

In [6]:
x = allkeywords.split()
# x

In [7]:
keywords_df = pd.DataFrame (x, columns = ['tag'])
print (keywords_df)

               tag
0             पैसा
1          रुपैयाँ
2         बिक्रीदर
3           खरिददर
4            कतारी
...            ...
761075       मतदान
761076     मन्त्री
761077      बोहरा,
761078  स्पष्टीकरण
761079     पटक–पटक

[761080 rows x 1 columns]


In [8]:
keywords_df = keywords_df.drop_duplicates().reset_index(drop=True)

In [9]:
keywords_df

,tag
0,पैसा
1,रुपैयाँ
2,बिक्रीदर
3,खरिददर
4,कतारी
...,...
39159,पठानसिंह
39160,लिँदैछन्।
39161,रामसरोज
39162,बन्दैछन्।


In [37]:
int(df[df['tag'] == 'पैसा'].index.values)

0

In [69]:
hostname = 'localhost'
username = 'ravi'
password = 'techprixa1234' # your password
database = 'news_recommendation_dev'

connection = psycopg2.connect(host=hostname, user=username, password=password, dbname=database)

In [82]:
def execute_values(conn, df, table):
  
    tuples = [tuple(x) for x in df.to_numpy()]
  
    cols = ','.join(list(df.columns))
    # SQL query to execute
    query = "INSERT INTO %s(%s) VALUES %%s" % (table, cols)
    cursor = conn.cursor()
    try:
        extras.execute_values(cursor, query, tuples)
        conn.commit()
    except (Exception, psycopg2.DatabaseError) as error:
        print("Error: %s" % error)
        conn.rollback()
        cursor.close()
        return 1
    print("the dataframe is inserted")
    cursor.close()

In [28]:
execute_values(connection, df, 'keywords')

the dataframe is inserted


In [49]:
len(pre.article_df)

38054

In [52]:
for i in tqdm(range(1000)):
    print(i)

100%|████████████████████████████████████| 1000/1000 [00:00<00:00, 20637.40it/s]

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [53]:
article_keyword_df = pd.DataFrame({'article_id': [], 'keywords_id': []})
article_keyword_df.keywords_id = article_keyword_df.keywords_id.astype(int)
for i in tqdm(range(len(pre.article_df))):
    article_id = pre.article_df.article_id.iloc[i]
    articles_keywords = pre.article_df.keywords_words.iloc[i].split()
    for keyword in articles_keywords:
        keyword_id = int(df[df['tag'] == keyword].index.values)
        article_keyword_df = article_keyword_df.append({'article_id':article_id, 'keywords_id':keyword_id}, ignore_index=True)


100%|███████████████████████████████████| 38054/38054 [1:34:08<00:00,  6.74it/s]


In [54]:
article_keyword_df

,article_id,keywords_id
0,ekuIAXIip3isHjFZAdmOs2rgb4KgToUZSIvURG0_AoA,0
1,ekuIAXIip3isHjFZAdmOs2rgb4KgToUZSIvURG0_AoA,1
2,ekuIAXIip3isHjFZAdmOs2rgb4KgToUZSIvURG0_AoA,2
3,ekuIAXIip3isHjFZAdmOs2rgb4KgToUZSIvURG0_AoA,3
4,ekuIAXIip3isHjFZAdmOs2rgb4KgToUZSIvURG0_AoA,4
...,...,...
761075,ujy8-UV0-DZoa3Lmx2qNZyOqEt754_4YkibDzKvadng,2672
761076,ujy8-UV0-DZoa3Lmx2qNZyOqEt754_4YkibDzKvadng,595
761077,ujy8-UV0-DZoa3Lmx2qNZyOqEt754_4YkibDzKvadng,27689
761078,ujy8-UV0-DZoa3Lmx2qNZyOqEt754_4YkibDzKvadng,7444


In [58]:
article_keyword_df.keywords_id = article_keyword_df['keywords_id'].apply(lambda x : x + 1)

In [59]:
execute_values(connection, article_keyword_df, 'article_keywords')

the dataframe is inserted


In [10]:
pre.user_df

,user,user_keywords
0,db783e901db42f2290736d314d08bdea,"[पैसा, रुपैयाँ, बिक्रीदर, खरिददर, कतारी, विनिम..."
1,83ba6e048bfb2245baeff4a8c40a7372,"[सुष्मिता, रोहमन, छुट्टी, तस्बिर, चुम्बन, प्रे..."
2,a704f987e0bc79f6bc185228f1cdfe8a,"[पक्ष, विवाद, पार्टी, अनुसार, संयुक्त, बैठक, आ..."
3,d9c3dae87d793c35fea9620a293962f4,"[विप्लव, वार्ता, युद्ध, जबरजस्त, धकेल्न, शान्त..."
4,a7c27e8ad54f246d33d92c85921c21eb,"[लागि, पार्टी, तयार, सरकार, प्रधानमन्त्री, दुर..."
...,...,...
185312,452aa355ab3ea4de4742e5ff36320eaf,"[भर्सन, फिमेल, सार्वजनिक, लामा, भने।, कम्पोज, ..."
185313,5cb16210f5bc5f18bb852ab846f3b5a2,"[खनाल–नेपाल, समूह, मन्त्रालय, मन्त्रिमण्डल, प्..."
185314,93c95988ec06dc1569a02ccb931c43a4,"[ज्ञानेन्द्र, भ्रम, राजा, अन्याय, भट्टराई, कटा..."
185315,433f43b6a40299d0a8c70351a2b57324,"[हजार, करोड, रसिया,, विश्व, कोरोना, ब्राजिल,, ..."


In [26]:
pre.user_df.user.unique()

array(['db783e901db42f2290736d314d08bdea',
       '83ba6e048bfb2245baeff4a8c40a7372',
       'a704f987e0bc79f6bc185228f1cdfe8a', ...,
       '93c95988ec06dc1569a02ccb931c43a4',
       '433f43b6a40299d0a8c70351a2b57324',
       '72afa487553978646c986b80c6970d80'], dtype=object)

In [11]:
from username_generator import username_generator

In [12]:
pre.user_df['username'] = pre.user_df.apply(lambda x : username_generator.generate_username(1)[0], axis=1)

In [13]:
pre.user_df

,user,user_keywords,username
0,db783e901db42f2290736d314d08bdea,"[पैसा, रुपैयाँ, बिक्रीदर, खरिददर, कतारी, विनिम...",verifiableRoutine338
1,83ba6e048bfb2245baeff4a8c40a7372,"[सुष्मिता, रोहमन, छुट्टी, तस्बिर, चुम्बन, प्रे...",imperturbableMan699
2,a704f987e0bc79f6bc185228f1cdfe8a,"[पक्ष, विवाद, पार्टी, अनुसार, संयुक्त, बैठक, आ...",someControl328
3,d9c3dae87d793c35fea9620a293962f4,"[विप्लव, वार्ता, युद्ध, जबरजस्त, धकेल्न, शान्त...",deadInternational571
4,a7c27e8ad54f246d33d92c85921c21eb,"[लागि, पार्टी, तयार, सरकार, प्रधानमन्त्री, दुर...",uprightLoad576
...,...,...,...
185312,452aa355ab3ea4de4742e5ff36320eaf,"[भर्सन, फिमेल, सार्वजनिक, लामा, भने।, कम्पोज, ...",frailMiddle721
185313,5cb16210f5bc5f18bb852ab846f3b5a2,"[खनाल–नेपाल, समूह, मन्त्रालय, मन्त्रिमण्डल, प्...",amazingMonth525
185314,93c95988ec06dc1569a02ccb931c43a4,"[ज्ञानेन्द्र, भ्रम, राजा, अन्याय, भट्टराई, कटा...",contentProfessional966
185315,433f43b6a40299d0a8c70351a2b57324,"[हजार, करोड, रसिया,, विश्व, कोरोना, ब्राजिल,, ...",well-to-doOutcome144


In [14]:
pre.user_df['device_name'] = "test_device"
pre.user_df['device_id'] = "test_id"
pre.user_df['ip_address'] = "test_ip"
pre.user_df['registered'] = True



In [15]:
pre.user_df

,user,user_keywords,username,device_name,device_id,ip_address,registered
0,db783e901db42f2290736d314d08bdea,"[पैसा, रुपैयाँ, बिक्रीदर, खरिददर, कतारी, विनिम...",verifiableRoutine338,test_device,test_id,test_ip,True
1,83ba6e048bfb2245baeff4a8c40a7372,"[सुष्मिता, रोहमन, छुट्टी, तस्बिर, चुम्बन, प्रे...",imperturbableMan699,test_device,test_id,test_ip,True
2,a704f987e0bc79f6bc185228f1cdfe8a,"[पक्ष, विवाद, पार्टी, अनुसार, संयुक्त, बैठक, आ...",someControl328,test_device,test_id,test_ip,True
3,d9c3dae87d793c35fea9620a293962f4,"[विप्लव, वार्ता, युद्ध, जबरजस्त, धकेल्न, शान्त...",deadInternational571,test_device,test_id,test_ip,True
4,a7c27e8ad54f246d33d92c85921c21eb,"[लागि, पार्टी, तयार, सरकार, प्रधानमन्त्री, दुर...",uprightLoad576,test_device,test_id,test_ip,True
...,...,...,...,...,...,...,...
185312,452aa355ab3ea4de4742e5ff36320eaf,"[भर्सन, फिमेल, सार्वजनिक, लामा, भने।, कम्पोज, ...",frailMiddle721,test_device,test_id,test_ip,True
185313,5cb16210f5bc5f18bb852ab846f3b5a2,"[खनाल–नेपाल, समूह, मन्त्रालय, मन्त्रिमण्डल, प्...",amazingMonth525,test_device,test_id,test_ip,True
185314,93c95988ec06dc1569a02ccb931c43a4,"[ज्ञानेन्द्र, भ्रम, राजा, अन्याय, भट्टराई, कटा...",contentProfessional966,test_device,test_id,test_ip,True
185315,433f43b6a40299d0a8c70351a2b57324,"[हजार, करोड, रसिया,, विश्व, कोरोना, ब्राजिल,, ...",well-to-doOutcome144,test_device,test_id,test_ip,True


In [16]:
from passlib.context import CryptContext

In [39]:
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

In [40]:
def get_password_hash(password):
    return pwd_context.hash(password)


In [41]:
hashed_password = get_password_hash("basicValuable222")

In [42]:
hashed_password

'$2b$12$bQeLWtJlWZZwqw1b6YZn9.NuIbH/4ZR79jFtpK0Vnqk.BDvy8kVre'

In [43]:
def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

In [44]:
verify_password('basicValuable222',hashed_password)

True

In [46]:
test_df = pre.user_df.head(5)

In [47]:
test_df

,user,user_keywords,username,device_name,device_id,ip_address,registered
0,db783e901db42f2290736d314d08bdea,"[पैसा, रुपैयाँ, बिक्रीदर, खरिददर, कतारी, विनिम...",basicValuable222,test_device,test_id,test_ip,True
1,83ba6e048bfb2245baeff4a8c40a7372,"[सुष्मिता, रोहमन, छुट्टी, तस्बिर, चुम्बन, प्रे...",insidiousProtection483,test_device,test_id,test_ip,True
2,a704f987e0bc79f6bc185228f1cdfe8a,"[पक्ष, विवाद, पार्टी, अनुसार, संयुक्त, बैठक, आ...",impressionableHeight454,test_device,test_id,test_ip,True
3,d9c3dae87d793c35fea9620a293962f4,"[विप्लव, वार्ता, युद्ध, जबरजस्त, धकेल्न, शान्त...",adeptPollution504,test_device,test_id,test_ip,True
4,a7c27e8ad54f246d33d92c85921c21eb,"[लागि, पार्टी, तयार, सरकार, प्रधानमन्त्री, दुर...",legalBody762,test_device,test_id,test_ip,True


In [50]:
test_df['password'] = test_df.apply(lambda  x : get_password_hash('password'), axis=1)

/tmp/ipykernel_116800/2333058586.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['password'] = test_df.apply(lambda  x : get_password_hash('password'), axis=1)


In [53]:
test_df['password'].iloc[0]

'$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJJCD0hlLJ7RblCS'

In [17]:
pre.user_df['password'] = '$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJJCD0hlLJ7RblCS'

In [18]:
pre.user_df['first_name'] = pre.user_df['username']

In [19]:
pre.user_df['last_name'] = "LastName"

In [20]:
pre.user_df['email'] = pre.user_df.apply(lambda  x : x['username']+'@gmail.com', axis=1)

In [21]:
pre.user_df

,user,user_keywords,username,device_name,device_id,ip_address,registered,password,first_name,last_name,email
0,db783e901db42f2290736d314d08bdea,"[पैसा, रुपैयाँ, बिक्रीदर, खरिददर, कतारी, विनिम...",verifiableRoutine338,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,verifiableRoutine338,LastName,verifiableRoutine338@gmail.com
1,83ba6e048bfb2245baeff4a8c40a7372,"[सुष्मिता, रोहमन, छुट्टी, तस्बिर, चुम्बन, प्रे...",imperturbableMan699,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,imperturbableMan699,LastName,imperturbableMan699@gmail.com
2,a704f987e0bc79f6bc185228f1cdfe8a,"[पक्ष, विवाद, पार्टी, अनुसार, संयुक्त, बैठक, आ...",someControl328,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,someControl328,LastName,someControl328@gmail.com
3,d9c3dae87d793c35fea9620a293962f4,"[विप्लव, वार्ता, युद्ध, जबरजस्त, धकेल्न, शान्त...",deadInternational571,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,deadInternational571,LastName,deadInternational571@gmail.com
4,a7c27e8ad54f246d33d92c85921c21eb,"[लागि, पार्टी, तयार, सरकार, प्रधानमन्त्री, दुर...",uprightLoad576,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,uprightLoad576,LastName,uprightLoad576@gmail.com
...,...,...,...,...,...,...,...,...,...,...,...
185312,452aa355ab3ea4de4742e5ff36320eaf,"[भर्सन, फिमेल, सार्वजनिक, लामा, भने।, कम्पोज, ...",frailMiddle721,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,frailMiddle721,LastName,frailMiddle721@gmail.com
185313,5cb16210f5bc5f18bb852ab846f3b5a2,"[खनाल–नेपाल, समूह, मन्त्रालय, मन्त्रिमण्डल, प्...",amazingMonth525,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,amazingMonth525,LastName,amazingMonth525@gmail.com
185314,93c95988ec06dc1569a02ccb931c43a4,"[ज्ञानेन्द्र, भ्रम, राजा, अन्याय, भट्टराई, कटा...",contentProfessional966,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,contentProfessional966,LastName,contentProfessional966@gmail.com
185315,433f43b6a40299d0a8c70351a2b57324,"[हजार, करोड, रसिया,, विश्व, कोरोना, ब्राजिल,, ...",well-to-doOutcome144,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,well-to-doOutcome144,LastName,well-to-doOutcome144@gmail.com


In [63]:
pre.user_df['profile_Image'] = "default_profile_pic.png"

In [64]:
pre.user_df

,user,user_keywords,username,device_name,device_id,ip_address,registered,password,first_name,last_name,email,profile_Image
0,db783e901db42f2290736d314d08bdea,"[पैसा, रुपैयाँ, बिक्रीदर, खरिददर, कतारी, विनिम...",basicValuable222,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,basicValuable222,LastName,basicValuable222@gmail.com,default_profile_pic.png
1,83ba6e048bfb2245baeff4a8c40a7372,"[सुष्मिता, रोहमन, छुट्टी, तस्बिर, चुम्बन, प्रे...",insidiousProtection483,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,insidiousProtection483,LastName,insidiousProtection483@gmail.com,default_profile_pic.png
2,a704f987e0bc79f6bc185228f1cdfe8a,"[पक्ष, विवाद, पार्टी, अनुसार, संयुक्त, बैठक, आ...",impressionableHeight454,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,impressionableHeight454,LastName,impressionableHeight454@gmail.com,default_profile_pic.png
3,d9c3dae87d793c35fea9620a293962f4,"[विप्लव, वार्ता, युद्ध, जबरजस्त, धकेल्न, शान्त...",adeptPollution504,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,adeptPollution504,LastName,adeptPollution504@gmail.com,default_profile_pic.png
4,a7c27e8ad54f246d33d92c85921c21eb,"[लागि, पार्टी, तयार, सरकार, प्रधानमन्त्री, दुर...",legalBody762,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,legalBody762,LastName,legalBody762@gmail.com,default_profile_pic.png
...,...,...,...,...,...,...,...,...,...,...,...,...
185312,452aa355ab3ea4de4742e5ff36320eaf,"[भर्सन, फिमेल, सार्वजनिक, लामा, भने।, कम्पोज, ...",achingSnipe338,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,achingSnipe338,LastName,achingSnipe338@gmail.com,default_profile_pic.png
185313,5cb16210f5bc5f18bb852ab846f3b5a2,"[खनाल–नेपाल, समूह, मन्त्रालय, मन्त्रिमण्डल, प्...",openPermit665,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,openPermit665,LastName,openPermit665@gmail.com,default_profile_pic.png
185314,93c95988ec06dc1569a02ccb931c43a4,"[ज्ञानेन्द्र, भ्रम, राजा, अन्याय, भट्टराई, कटा...",immediateDay750,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,immediateDay750,LastName,immediateDay750@gmail.com,default_profile_pic.png
185315,433f43b6a40299d0a8c70351a2b57324,"[हजार, करोड, रसिया,, विश्व, कोरोना, ब्राजिल,, ...",madCoffee876,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,madCoffee876,LastName,madCoffee876@gmail.com,default_profile_pic.png


In [66]:
pre.user_df.columns

Index(['user', 'user_keywords', 'username', 'device_name', 'device_id',
       'ip_address', 'registered', 'password', 'first_name', 'last_name',
       'email', 'profile_Image'],
      dtype='object')

In [22]:
insert_user_df = pre.user_df[['user', 'username', 'device_name', 'device_id',
       'ip_address', 'registered', 'password', 'first_name', 'last_name',
       'email']]

In [89]:
insert_user_df = pre.user_df[['user', 'username', 'device_name', 'device_id',
       'ip_address', 'registered']]

In [23]:
insert_user_df.rename(columns={"user": "id"},inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_11936\2876636026.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  insert_user_df.rename(columns={"user": "id"},inplace=True)


In [24]:
insert_user_df

,id,username,device_name,device_id,ip_address,registered,password,first_name,last_name,email
0,db783e901db42f2290736d314d08bdea,verifiableRoutine338,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,verifiableRoutine338,LastName,verifiableRoutine338@gmail.com
1,83ba6e048bfb2245baeff4a8c40a7372,imperturbableMan699,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,imperturbableMan699,LastName,imperturbableMan699@gmail.com
2,a704f987e0bc79f6bc185228f1cdfe8a,someControl328,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,someControl328,LastName,someControl328@gmail.com
3,d9c3dae87d793c35fea9620a293962f4,deadInternational571,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,deadInternational571,LastName,deadInternational571@gmail.com
4,a7c27e8ad54f246d33d92c85921c21eb,uprightLoad576,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,uprightLoad576,LastName,uprightLoad576@gmail.com
...,...,...,...,...,...,...,...,...,...,...
185312,452aa355ab3ea4de4742e5ff36320eaf,frailMiddle721,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,frailMiddle721,LastName,frailMiddle721@gmail.com
185313,5cb16210f5bc5f18bb852ab846f3b5a2,amazingMonth525,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,amazingMonth525,LastName,amazingMonth525@gmail.com
185314,93c95988ec06dc1569a02ccb931c43a4,contentProfessional966,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,contentProfessional966,LastName,contentProfessional966@gmail.com
185315,433f43b6a40299d0a8c70351a2b57324,well-to-doOutcome144,test_device,test_id,test_ip,True,$2b$12$CAvH2szhJB7k64d7ClCU5OJUznjzLTiSIQLZCBJ...,well-to-doOutcome144,LastName,well-to-doOutcome144@gmail.com


In [81]:
insert_user_df.columns

Index(['id', 'username', 'device_name', 'device_id', 'ip_address',
       'registered', 'password', 'first_name', 'last_name', 'email',
       'profile_Image'],
      dtype='object')

In [25]:
hostname = 'localhost'
username = 'postgres'
password = 'techprixa1234' # your password
database = 'news_recommendation_dev'

connection = psycopg2.connect(host=hostname, user=username, password=password, dbname=database)

In [26]:
def execute_values(conn, df, table):
  
    tuples = [tuple(x) for x in df.to_numpy()]
    
    cols = ','.join(list(df.columns))
    # SQL query to execute
    print(f"columns are {cols}")
    query = 'INSERT INTO "%s" (%s) VALUES %%s' % (table, cols)
    print(f"query IS {query}")
    cursor = conn.cursor()
    try:
        extras.execute_values(cursor, query, tuples)
        conn.commit()
    except (Exception, psycopg2.DatabaseError) as error:
        print("Error: %s" % error)
        conn.rollback()
        cursor.close()
        return 1
    print("the dataframe is inserted")
    cursor.close()

In [27]:
execute_values(connection, insert_user_df,"user")

columns are id,username,device_name,device_id,ip_address,registered,password,first_name,last_name,email
query IS INSERT INTO "user" (id,username,device_name,device_id,ip_address,registered,password,first_name,last_name,email) VALUES %s
the dataframe is inserted


In [8]:
type(pre.user_df.user_keywords.iloc[0])

list

In [9]:
pre.user_df.user.iloc[0]

'db783e901db42f2290736d314d08bdea'

In [22]:
pre.user_df.iloc['d9c3dae87d793c35fea9620a293962f4']

TypeError: Cannot index by location index with a non-integer key

In [15]:
class Engine(object):
    def __init__(self, keywords_df, user_keyword_df ):
        self.keywords_df = keywords_df
        self.user_keyword_df = user_keyword_df
    def __call__(self, idx):
#         print(f"processing index: {idx}")
        user_id = pre.user_df.user.iloc[idx]
        user_keywords = pre.user_df.user_keywords.iloc[idx]
        for keyword in user_keywords:
            keyword_id = int(self.keywords_df[self.keywords_df['tag'] == keyword].index.values)
            self.user_keyword_df = self.user_keyword_df.append({'user_id':user_id, 'keywords_id':keyword_id}, ignore_index=True)
            
        
        print(self.user_keyword_df)

In [12]:
import time

In [13]:
user_keyword_df = pd.DataFrame({'user_id': [], 'keywords_id': []})
user_keyword_df.keywords_id = user_keyword_df.keywords_id.astype(int)

In [19]:
final_user_keyword_df

,user_id,keywords_id


In [18]:
try:
    final_user_keyword_df = pd.DataFrame({'user_id': [], 'keywords_id': []})
    start_time = time.time()
    print(f'starting process at {start_time}')
    pool = Pool(8) # on 8 processors
    engine = Engine(keywords_df,user_keyword_df)
#     data_outputs = pool.map(engine,range(len(pre.user_df)))
    data_outputs = pool.map(engine,range(5))
    end_time = time.time()
    print(f"ending process at {end_time}")
    print(f"total time taken: {end_time - start_time}")
    
finally: # To make sure processes are closed in the end, even if errors happen
    final_user_keyword_df.append(engine.user_keyword_df, ignore_index=True )
#     final_user_keyword_df = engine.user_keyword_df
    pool.close()
    pool.join()

starting process at 1655187580.2740066
                             user_id  keywords_id
0   83ba6e048bfb2245baeff4a8c40a7372           20
1   83ba6e048bfb2245baeff4a8c40a7372           21
2   83ba6e048bfb2245baeff4a8c40a7372           22
3   83ba6e048bfb2245baeff4a8c40a7372           23
4   83ba6e048bfb2245baeff4a8c40a7372           24
5   83ba6e048bfb2245baeff4a8c40a7372           25
6   83ba6e048bfb2245baeff4a8c40a7372           26
7   83ba6e048bfb2245baeff4a8c40a7372           27
8   83ba6e048bfb2245baeff4a8c40a7372           28
9   83ba6e048bfb2245baeff4a8c40a7372           29
10  83ba6e048bfb2245baeff4a8c40a7372           30
11  83ba6e048bfb2245baeff4a8c40a7372           31
12  83ba6e048bfb2245baeff4a8c40a7372           32
13  83ba6e048bfb2245baeff4a8c40a7372           33
14  83ba6e048bfb2245baeff4a8c40a7372           34
15  83ba6e048bfb2245baeff4a8c40a7372           35
16  83ba6e048bfb2245baeff4a8c40a7372           36
17  83ba6e048bfb2245baeff4a8c40a7372           37
18  83ba6e0

In [19]:
try:
    start_time = time.time()
    print(f'starting process at {start_time}')
    pool = Pool(8) # on 8 processors
    engine = Engine(keywords_df,user_keyword_df)
    data_outputs = pool.map(engine,range(len(pre.user_df)))
    end_time = time.time()
    print(f"ending process at {end_time}")
    print(f"total time taken: {end_time - start_time}")
    
finally: # To make sure processes are closed in the end, even if errors happen
    
#     final_user_keyword_df = engine.user_keyword_df
    pool.close()
    pool.join()

starting process at 1655184831.1164653
ending process at 1655186868.6998465
total time taken: 2037.5833811759949


In [20]:
engine.user_keyword_df

,user_id,keywords_id


In [21]:
user_keyword_df

,user_id,keywords_id


In [29]:
from multiprocessing import Pool

In [22]:

for i in tqdm(range(len(pre.user_df))):
    user_id = pre.user_df.user.iloc[i]
    user_keywords = pre.user_df.user_keywords.iloc[i]
    for keyword in user_keywords:
        keyword_id = int(keywords_df[keywords_df['tag'] == keyword].index.values)
        user_keyword_df = user_keyword_df.append({'user_id':user_id, 'keywords_id':keyword_id}, ignore_index=True)


  0%|                                    | 152/185317 [00:11<3:47:08, 13.59it/s]


KeyboardInterrupt: 